# Nemotron SFT Smoke Run — v1.1 traces — no TRL dependency

Purpose:
- consume `train_traces_v1_1.jsonl` from Notebook A
- run a **tiny manual PEFT/LoRA smoke update**
- avoid `trl` / `SFTTrainer` because the pinned Kaggle environment may not include TRL and internet is off
- prove adapter training step + adapter save path

This is **not** leaderboard optimization and **not** final submission packaging.


In [ ]:
import gc
import json
import os
import random
import re
import shutil
import stat
import sys
import zipfile
from pathlib import Path

import pandas as pd

TRACE_VERSION = "v1_1"
SMOKE_SAMPLE_SIZE = 12      # intentionally tiny; increase only after the smoke path succeeds
RANDOM_SEED = 42
MAX_SEQ_LEN = 512           # keep short to reduce activation memory
LR = 2e-4

LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORKING_DIR / "adapter_smoke_v1_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACE_JSONL_CANDIDATES = [WORKING_DIR / "train_traces_v1_1.jsonl"]
if Path("/kaggle/input").exists():
    TRACE_JSONL_CANDIDATES.extend(Path("/kaggle/input").rglob("train_traces_v1_1.jsonl"))

TRACE_JSONL_PATH = next((p for p in TRACE_JSONL_CANDIDATES if p.exists()), None)
if TRACE_JSONL_PATH is None:
    print("Known trace candidates:")
    for p in TRACE_JSONL_CANDIDATES[:50]:
        print(" -", p)
    raise FileNotFoundError("Could not find train_traces_v1_1.jsonl. Run Notebook A or add its output as input.")

print("TRACE_JSONL_PATH:", TRACE_JSONL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Imports

This version deliberately avoids `trl` and `datasets`. It uses a minimal manual training loop with PEFT.


In [ ]:
import torch
import torch.nn.functional as F

import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Runtime patches for Nemotron/Mamba/Triton robustness

In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        try:
            mod.rmsnorm_fn = _pure_rmsnorm_fn
        except Exception:
            pass

PTXAS_BLACKWELL_CANDIDATES = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"),
    Path("/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"),
]
ptxas_src = next((p for p in PTXAS_BLACKWELL_CANDIDATES if p.exists()), None)
if ptxas_src is not None:
    dst = Path("/tmp/ptxas-blackwell")
    shutil.copy2(ptxas_src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = str(dst)
    print("Copied ptxas-blackwell to:", dst)
else:
    print("ptxas-blackwell utility binary not found; continuing.")

try:
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
    print("Patched Triton get_ptxas_version.")
except Exception as e:
    print("Could not patch Triton compiler version:", repr(e))


## Load v1.1 trace corpus and build a tiny stratified smoke sample

In [ ]:
records = []
with open(TRACE_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

assert records, "No records found."
df = pd.DataFrame(records)
print("Total trace records:", len(df))
print(df[["id", "category", "answer", "approx_trace_chars"]].head())
print("\nCategory counts:")
print(df["category"].value_counts())

per_category = max(1, SMOKE_SAMPLE_SIZE // df["category"].nunique())
sampled = (
    df.groupby("category", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), per_category), random_state=RANDOM_SEED))
      .sample(frac=1.0, random_state=RANDOM_SEED)
      .head(SMOKE_SAMPLE_SIZE)
      .reset_index(drop=True)
)

print("\nSmoke sample shape:", sampled.shape)
print(sampled["category"].value_counts())


In [ ]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print("MODEL_PATH:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_training_text(row):
    messages = row["messages"]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=True,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        user = messages[0]["content"]
        assistant = messages[1]["content"]
        return f"User:\n{user}\n\nAssistant:\n{assistant}{tokenizer.eos_token}"

sampled["text"] = sampled.apply(build_training_text, axis=1)
print("Example text length:", len(sampled.loc[0, "text"]))
print(sampled.loc[0, "text"][:1200])


## Load model and attach LoRA

For smoke robustness, this tries full CUDA placement first on the 96GB Kaggle GPU. If this fails, capture the exact error; do not repeatedly retry the same cell.


In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        try:
            mod.is_fast_path_available = False
            print(f"Patched {name}: is_fast_path_available = False")
        except Exception:
            pass

model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.train()


## Tiny manual SFT update

This trains on 12 examples, one optimizer step per example. The goal is only to verify that backward + optimizer + adapter save work.


In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
assert trainable_params, "No trainable LoRA parameters found. Check LORA_TARGET_MODULES."

optimizer = torch.optim.AdamW(trainable_params, lr=LR)
losses = []

for step, text in enumerate(sampled["text"].tolist(), start=1):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    labels = enc["input_ids"].clone()

    optimizer.zero_grad(set_to_none=True)
    out = model(**enc, labels=labels)
    loss = out.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    loss_value = float(loss.detach().cpu())
    losses.append(loss_value)
    print(f"step {step:02d}/{len(sampled)} | loss={loss_value:.4f}")

print("Manual SFT smoke loop complete.")
print("mean_loss:", sum(losses) / len(losses))


## Save adapter artifacts and validate folder

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer_debug")

print("Adapter output files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR), f"({p.stat().st_size/1024:.1f} KB)")

adapter_config = OUTPUT_DIR / "adapter_config.json"
adapter_safetensors = OUTPUT_DIR / "adapter_model.safetensors"
adapter_bin = OUTPUT_DIR / "adapter_model.bin"

assert adapter_config.exists(), "Missing adapter_config.json"
assert adapter_safetensors.exists() or adapter_bin.exists(), "Missing adapter model weights"

print("\nSmoke adapter save validation passed.")
print(OUTPUT_DIR)


## Optional debug zip

This zip is for moving the smoke adapter to a validation notebook. It is **not** final submission packaging.


In [ ]:
SMOKE_ZIP_PATH = WORKING_DIR / "adapter_smoke_v1_1.zip"

with zipfile.ZipFile(SMOKE_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name.startswith("adapter_"):
            zf.write(p, arcname=p.name)

print("Created:", SMOKE_ZIP_PATH, f"({SMOKE_ZIP_PATH.stat().st_size/1024/1024:.2f} MB)")
with zipfile.ZipFile(SMOKE_ZIP_PATH, "r") as zf:
    print("Zip contents:", zf.namelist())
    assert "adapter_config.json" in zf.namelist()
